# Hospital Patient Readmission Predictor

## Reading the EHR data

In [9]:
import pandas as pd
import numpy as np
from datetime import timedelta
import sqlite3

### Importing data from csv files

In [10]:
patients=pd.read_csv('data/patients.csv')
admissions=pd.read_csv('data/admissions.csv')
labs=pd.read_csv('data/labs.csv')

In [11]:
patients.sample(2)

,patient_id,age,gender,blood_group,insurance_type,district,chronic_conditions
148140,P148140,54,F,AB-,private,urban,3
426512,P426512,22,M,AB+,private,rural,1


In [12]:
patients.shape

(775600, 7)

In [13]:
patients.nunique()

patient_id            775600
age                       72
gender                     3
blood_group                8
insurance_type             3
district                   3
chronic_conditions         5
dtype: int64

In [14]:
patients.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 775600 entries, 0 to 775599
Data columns (total 7 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   patient_id          775600 non-null  object
 1   age                 775600 non-null  int64 
 2   gender              775600 non-null  object
 3   blood_group         775600 non-null  object
 4   insurance_type      775600 non-null  object
 5   district            775600 non-null  object
 6   chronic_conditions  775600 non-null  int64 
dtypes: int64(2), object(5)
memory usage: 41.4+ MB


In [15]:
admissions.sample(2)

,admission_id,patient_id,admission_date,department,admission_type,length_of_stay_days,num_procedures,num_medications_discharge,discharge_disposition
111969,A111969,P63745,2025-07-05,endocrinology,emergency,2,3,6,home
779963,A779963,P651681,2025-03-17,cardiology,elective,1,1,8,home_health_service


In [16]:
admissions.shape

(1000000, 9)

In [17]:
admissions.nunique()

admission_id                 1000000
patient_id                    562002
admission_date                   731
department                         6
admission_type                     3
length_of_stay_days               45
num_procedures                    13
num_medications_discharge         11
discharge_disposition              4
dtype: int64

In [18]:
admissions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 9 columns):
 #   Column                     Non-Null Count    Dtype 
---  ------                     --------------    ----- 
 0   admission_id               1000000 non-null  object
 1   patient_id                 1000000 non-null  object
 2   admission_date             1000000 non-null  object
 3   department                 1000000 non-null  object
 4   admission_type             1000000 non-null  object
 5   length_of_stay_days        1000000 non-null  int64 
 6   num_procedures             1000000 non-null  int64 
 7   num_medications_discharge  1000000 non-null  int64 
 8   discharge_disposition      1000000 non-null  object
dtypes: int64(3), object(6)
memory usage: 68.7+ MB


In [19]:
labs.sample(2)

,admission_id,hemoglobin,wbc_count,creatinine,glucose_random,sodium,potassium
326985,A326985,12.0,8.2,2.65,113.0,142.0,4.1
955704,A955704,16.8,11.6,2.82,251.0,140.0,4.2


In [20]:
labs.shape

(1000000, 7)

In [21]:
labs.nunique()

admission_id      1000000
hemoglobin            151
wbc_count             281
creatinine            740
glucose_random        307
sodium                 38
potassium              46
dtype: int64

In [22]:
labs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   admission_id    1000000 non-null  object 
 1   hemoglobin      920571 non-null   float64
 2   wbc_count       1000000 non-null  float64
 3   creatinine      920390 non-null   float64
 4   glucose_random  920045 non-null   float64
 5   sodium          1000000 non-null  float64
 6   potassium       1000000 non-null  float64
dtypes: float64(6), object(1)
memory usage: 53.4+ MB


In [23]:
labs.isnull().mean()*100

admission_id      0.0000
hemoglobin        7.9429
wbc_count         0.0000
creatinine        7.9610
glucose_random    7.9955
sodium            0.0000
potassium         0.0000
dtype: float64

### Merging all the imported files into a single df

In [24]:
df = admissions.merge(patients,on='patient_id').merge(labs,on='admission_id')

In [25]:
df.sample(2)

,admission_id,patient_id,admission_date,department,admission_type,length_of_stay_days,num_procedures,num_medications_discharge,discharge_disposition,age,...,blood_group,insurance_type,district,chronic_conditions,hemoglobin,wbc_count,creatinine,glucose_random,sodium,potassium
990580,A990580,P64107,2025-10-18,pulmonology,elective,4,2,2,home,65,...,B-,private,rural,3,11.0,11.8,0.30,206.0,136.0,3.2
399782,A399782,P706500,2025-08-11,general_medicine,urgent,4,4,7,rehab,63,...,AB-,government,urban,2,12.4,5.9,2.21,NaN,140.0,4.4


In [26]:
df.shape

(1000000, 21)

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 21 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   admission_id               1000000 non-null  object 
 1   patient_id                 1000000 non-null  object 
 2   admission_date             1000000 non-null  object 
 3   department                 1000000 non-null  object 
 4   admission_type             1000000 non-null  object 
 5   length_of_stay_days        1000000 non-null  int64  
 6   num_procedures             1000000 non-null  int64  
 7   num_medications_discharge  1000000 non-null  int64  
 8   discharge_disposition      1000000 non-null  object 
 9   age                        1000000 non-null  int64  
 10  gender                     1000000 non-null  object 
 11  blood_group                1000000 non-null  object 
 12  insurance_type             1000000 non-null  object 
 13  district     

### Creating Readmission Score

In [28]:
n_admissions=df['admission_id'].count()
n_admissions

np.int64(1000000)

In [29]:
readmit_score = (
    (df['age']>65).astype(float)*1.5 +
    (df['chronic_conditions']>=2).astype(float)*2+
    (df['length_of_stay_days']>7).astype(float)*1.5+
    (df['admission_type']=='emergency').astype(float)*1.5+
    (df['num_medications_discharge']>6).astype(float)*1+
    (df['discharge_disposition']!='home').astype(float)*1.5+
    (df['creatinine']>2).astype(float)*1+
    (df['insurance_type']=='self_pay').astype(float)*0.5+
    (df['district']=='rural').astype(float)*0.5+
    np.random.normal(0,2,n_admissions)
)

In [30]:
df['readmitted_30_days']=(readmit_score>5).astype(int)

In [31]:
df.sample(2)

,admission_id,patient_id,admission_date,department,admission_type,length_of_stay_days,num_procedures,num_medications_discharge,discharge_disposition,age,...,insurance_type,district,chronic_conditions,hemoglobin,wbc_count,creatinine,glucose_random,sodium,potassium,readmitted_30_days
384711,A384711,P138471,2025-03-08,general_medicine,urgent,2,2,2,rehab,79,...,private,urban,0,14.8,8.6,NaN,172.0,138.0,3.5,1
922889,A922889,P382803,2025-08-16,endocrinology,elective,7,5,5,home,83,...,government,rural,3,12.1,5.5,1.86,158.0,142.0,4.5,0


In [32]:
print(f"Total admissions : {n_admissions}, Readmission rate: {df['readmitted_30_days'].mean():.1%}")

Total admissions : 1000000, Readmission rate: 29.6%


## SQL EDA

In [33]:
import sqlite3
conn = sqlite3.connect('data/hospital.db')
patients.to_sql('patients',conn,if_exists='replace',index=False)
admissions.to_sql('admissions',conn,if_exists='replace',index=False)
labs.to_sql('labs',conn,if_exists='replace',index=False)

1000000

In [34]:
query="""
    select 
    P.district,
    P.insurance_type,
    A.department,
    count(A.admission_id) as Total_Admission,
    round(avg(A.length_of_stay_days),2) as avg_los
    from admissions as A
    join patients as P on A.patient_id=P.patient_id
    join labs as L on A.admission_id = L.admission_id
    where A.admission_type='emergency'
    group by
    P.district,
    P.insurance_type,
    A.department
    having count(A.admission_id)>10000
    
"""

In [35]:
pd.read_sql(query,conn)

,district,insurance_type,department,Total_Admission,avg_los
0,rural,private,general_medicine,10063,3.82
1,semi_urban,government,general_medicine,12223,3.78
2,semi_urban,private,cardiology,11100,3.74
3,semi_urban,private,general_medicine,14154,3.76
4,urban,government,cardiology,11069,3.78
5,urban,government,general_medicine,13933,3.75
6,urban,private,cardiology,12826,3.80
7,urban,private,general_medicine,16189,3.76
8,urban,self_pay,general_medicine,10064,3.80


## Feature Engineering

In [36]:
from sklearn.preprocessing import LabelEncoder

In [37]:
# Time based features
df['admission_month'] = pd.to_datetime(df['admission_date']).dt.month
df['is_weekend_admission'] = pd.to_datetime(df['admission_date']).dt.dayofweek>=5

In [38]:
# Clinical severity indicator
df['abnormal_creatinine']=(df['creatinine'] > 1.5).astype(int)
df['abnormal_wbc']=((df['wbc_count']<4)|(df['wbc_count']>11)).astype(int)
df['anemia']=(df['hemoglobin']<10).astype(int)
df['hyperglycemia']=(df['glucose_random'] > 200).astype(int)
df['abnormal_sodium']=((df['sodium']<135)|(df['sodium']>145)).astype(int)
df['total_abnormal_labs']=(df['abnormal_creatinine']+df['abnormal_wbc']+df['anemia']+df['hyperglycemia']+df['abnormal_sodium'])

In [39]:
# Patient History
patient_history = df.groupby('patient_id').agg(
    prior_admissions=('admission_id','count'),
    avg_prior_los=('length_of_stay_days','mean'),
    max_prior_medications=('num_medications_discharge','max'),
).reset_index()
df=df.merge(patient_history,on='patient_id',suffixes=('','_hist'))

In [40]:
# Interaction Features
df['age_X_chronic'] = df['age']*df['chronic_conditions']
df['los_x_meds'] = df['length_of_stay_days']*df['num_medications_discharge']

In [41]:
# Encoding Categorical Features
cat_cols=['gender','department','admission_type','discharge_disposition','insurance_type','district','blood_group']
for col in cat_cols:
    df[col]=LabelEncoder().fit_transform(df[col])
print(f"Total Features : {len(df.columns)-3}")

Total Features : 32


## LigthGBM algorithm based Model with MLFlow tracking

In [42]:
!pip install lightgbm
!pip install mlflow

In [43]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split,StratifiedKFold
from sklearn.metrics import roc_auc_score,classification_report,precision_recall_curve
from sklearn.impute import SimpleImputer
import mlflow
import mlflow.sklearn

### Preparing the data

In [44]:
drop_cols = ['admission_id','patient_id','admission_date','readmitted_30_days']
feature_cols = [c for c in df.columns if c not in drop_cols]

In [45]:
X = df[feature_cols].copy()
y = df['readmitted_30_days']

### Impute missing values

In [46]:
imputer = SimpleImputer(strategy='median')
X_imputed = pd.DataFrame(imputer.fit_transform(X),columns=X.columns)

In [47]:
X_train,X_test,y_train,y_test = train_test_split(X_imputed,y,test_size=0.2,random_state=42,stratify=y)

### MLFlow Experimentation

In [48]:
mlflow.set_experiment("Hospital_Readmission")
params_list = [
    {'n_estimators':200,'max_depth':5,'learning_rate':0.1,'num_leaves':31},
    {'n_estimators':300,'max_depth':7,'learning_rate':0.05,'num_leaves':63},
    {'n_estimators':500,'max_depth':6,'learning_rate':0.03,'num_leaves':50},
]

2026/05/28 11:48:49 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/05/28 11:48:50 INFO mlflow.store.db.utils: Updating database tables
2026/05/28 11:48:52 INFO mlflow.tracking.fluent: Experiment with name 'Hospital_Readmission' does not exist. Creating a new experiment.


In [52]:
best_auc = 0
for i,params in enumerate(params_list):
    with mlflow.start_run(run_name=f"lgbm_run_{i+1}"):
        model = lgb.LGBMClassifier(**params,random_state=42,class_weight='balanced',verbose=-1,metric='auc')
        model.fit(X_train,y_train,eval_set=[(X_test,y_test)],callbacks=[lgb.early_stopping(50),lgb.log_evaluation(0)])
        y_proba = model.predict_proba(X_test)[:,1]
        auc=roc_auc_score(y_test,y_proba)
        
        mlflow.log_params(params)
        mlflow.log_metric("auc",auc)
        mlflow.sklearn.log_model(model, "model")

        print(f"Run {1+1}: AUC = {auc:.4f}")
        if auc > best_auc:
            best_auc = auc
            best_model = model
print(f"Best model:{best_model}")

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[99]	valid_0's auc: 0.82794


2026/05/28 14:32:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 14:32:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2: AUC = 0.8279
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[161]	valid_0's auc: 0.828018


2026/05/28 14:33:05 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 14:33:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2: AUC = 0.8280
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[252]	valid_0's auc: 0.827965


2026/05/28 14:33:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/28 14:33:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Run 2: AUC = 0.8280
Best model:LGBMClassifier(class_weight='balanced', learning_rate=0.05, max_depth=7,
               metric='auc', n_estimators=300, num_leaves=63, random_state=42,
               verbose=-1)


In [64]:
# Saving the model
import os
os.makedirs('models',exist_ok=True)
joblib.dump(best_model,'models/readmission_lgbm.pkl')

['models/readmission_lgbm.pkl']

## Fairness analysis

### Check if the model is biased across demographic groups

In [54]:
# Restore original categorical values for analysis
df_test = X_test.copy()
df_test['y_true'] = y_test.values
df_test['y_proba'] = best_model.predict_proba(X_test)[:,1]
df_test['y_pred'] = (df_test['y_proba'] > 0.5).astype(int)

In [56]:
# Fairness by gender (0=F, 1=M, 2=Other)
print("===== Fairness : Gender =====")
for gender_val in df_test['gender'].unique():
    subset = df_test[df_test['gender']==gender_val]
    if len(subset)>50:
        auc = roc_auc_score(subset['y_true'],subset['y_proba'])
        fpr = (subset['y_pred'].sum() - (subset['y_true'] & subset['y_pred']).sum())/(~subset['y_true'].astype(bool)).sum()
        print(f" Gender = {gender_val} : AUC = {auc:.3f} , N = {len(subset)}")

===== Fairness : Gender =====
 Gender = 1.0 : AUC = 0.828 , N = 96185
 Gender = 0.0 : AUC = 0.829 , N = 99715
 Gender = 2.0 : AUC = 0.815 , N = 4100


In [57]:
# Fairness by Insurance type
print("\n ===== Fairness : Insurance =====")
for ins_val in df_test['insurance_type'].unique():
    subset = df_test[df_test['insurance_type']==ins_val]
    if len(subset) > 50:
        auc = roc_auc_score(subset['y_true'],subset['y_proba'])
        print(f" Insurance={ins_val} : AUC={auc:.3f} , N = {len(subset)}")


 ===== Fairness : Insurance =====
 Insurance=1.0 : AUC=0.830 , N = 80002
 Insurance=0.0 : AUC=0.828 , N = 70034
 Insurance=2.0 : AUC=0.819 , N = 49964


In [59]:
# Fairness by age group
print("\n ===== Fairness: Age =====")
for label,(lo,hi) in {'18-39':(18,40),'40-59':(40,60),'60+  ':(60,100)}.items():
    subset = df_test[(df_test['age']>=lo) & (df_test['age']<hi)]
    if len(subset) > 50:
        auc = roc_auc_score(subset['y_true'],subset['y_proba'])
        print(f" {label}: AUC={auc:.3f}, N={len(subset)}")


 ===== Fairness: Age =====
 18-39: AUC=0.825, N=61412
 40-59: AUC=0.817, N=55806
 60+  : AUC=0.816, N=82782


## SHAP and Streamlit Clinical Dashboard

In [68]:
%%writefile readmission_app.py
import streamlit as st
import shap
import joblib
import pandas as pd
import matplotlib.pyplot as plt

st.set_page_config(page_title="Readmission Risk",page_icon = "🏥",layout="wide")
st.title("🏥 30-Day Readmission Risk Dashboard")

model=joblib.load('models/readmission_lgbm.pkl')
explainer=shap.TreeExplainer(model)

# Patient input form
with st.form("patient_form"):
    col1,col2,col3 = st.columns(3)
    with col1:
        age = st.number_input("Age",18,90,65)
        chronic = st.selectbox("Chronic condition",[0,1,2,3,4])
        los = st.number_input("Length od Stay (days)",1,45,5)
    with col2:
        dept = st.selectbox("Department",['cardiology','pulmonology','general_medicine'])
        admit_type = st.selectbox("Admission Type",['emergency','elective','urgent'])
        n_meds = st.number_input("Medications at discharge",1,12,4)
    with col3:
        creatinine = st.number_input("Creatinine",0.3,12.0,1.0)
        hemoglobin = st.number_input("Hemoglobin",5.0,20.0,12.5)
        discharge = st.selectbox("Discharge To",['home','home_health_service','rehab'])

    submitted=st.form_submit_button("Calculate Risk",type="primary")

if submitted:
    risk_score=0.35 #placeholder
    st.metric("Readmission Risk",f"{risk_score:.0%}",delta="HIGH RISK" if risk_score > 0.4 else "Low Risk")

Overwriting readmission_app.py


In [69]:
!streamlit run readmission_app.py

^C
